In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rl-fundamentals",
    notebook_name="04_policies_and_value_functions_experiments.ipynb"
)

# Experiments: Policies and Value Functions

This notebook provides runnable evidence for claims about value functions, policies, and the advantage function.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Experiment 1: Complexity Benchmark — Tabular Q Memory vs State-Action Space

**Claim:** Tabular Q requires O(|S|*|A|) memory, which becomes impractical for large state spaces.

**Why it matters in an interview:** This motivates function approximation (neural networks) as a necessity, not a preference.

In [ ]:
import sys

# Measure actual memory for different Q-table sizes
configs = [
    (10, 4, "Small grid"),
    (100, 4, "10x10 grid"),
    (1000, 4, "~32x32 grid"),
    (10000, 4, "100x100 grid"),
    (100000, 4, "~316x316 grid"),
    (1000000, 4, "1000x1000 grid"),
    (10000000, 4, "~3162x3162 grid"),
]

print("Q-table memory requirements (float64)")
print(f"{'|S|':>12} | {'|A|':>5} | {'Entries':>12} | {'Memory':>12} | {'Description':>20}")
print("-" * 75)

sizes_for_plot = []
memory_mb = []

for n_states, n_actions, desc in configs:
    entries = n_states * n_actions
    bytes_needed = entries * 8  # float64 = 8 bytes
    
    if bytes_needed < 1024:
        mem_str = f"{bytes_needed} B"
    elif bytes_needed < 1024**2:
        mem_str = f"{bytes_needed/1024:.1f} KB"
    elif bytes_needed < 1024**3:
        mem_str = f"{bytes_needed/1024**2:.1f} MB"
    else:
        mem_str = f"{bytes_needed/1024**3:.2f} GB"
    
    sizes_for_plot.append(n_states)
    memory_mb.append(bytes_needed / 1024**2)
    
    print(f"{n_states:>12,} | {n_actions:>5} | {entries:>12,} | {mem_str:>12} | {desc:>20}")

# Now show why Atari is impossible
print("\n--- Why tabular methods fail for Atari ---")
atari_states = 256 ** (84 * 84)  # 84x84 pixels, 256 grayscale values each
print(f"Atari state space (84x84 grayscale): 256^{84*84} ≈ 10^{84*84*np.log10(256):.0f} states")
print(f"Even with 4 frames stacked: 256^{84*84*4} ≈ 10^{84*84*4*np.log10(256):.0f} states")
print(f"A Q-table for this would need more entries than atoms in the universe (~10^80).")
print(f"\nThis is why DQN uses a neural network instead of a table.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(sizes_for_plot, memory_mb, 'bo-', linewidth=2, markersize=8)
ax.axhline(y=1024, color='red', linestyle='--', label='1 GB limit', linewidth=2)
ax.axhline(y=16*1024, color='darkred', linestyle='--', label='16 GB limit', linewidth=2)
ax.set_xlabel('Number of States |S|')
ax.set_ylabel('Memory (MB, log scale)')
ax.set_title('Q-Table Memory: Linear in |S|, Impractical Above ~10M States')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Interview sentence: 'Tabular Q stores one float per state-action pair.")
print("A 1000x1000 grid with 4 actions needs 32 MB — manageable. Atari with")
print("84x84x4 pixel inputs has 10^17000 states — tabular methods are impossible.")
print("This is why function approximation (DQN) is necessary.'")

## Experiment 2: Failure Mode — Overestimation Bias in Q-Learning

**Claim:** Standard Q-learning systematically overestimates Q-values because max over noisy estimates introduces a positive bias.

**Why it matters in an interview:** This is the key problem that Double DQN fixes. Understanding the bias mechanism is expected at the staff level.

In [ ]:
class SimpleChainEnv:
    """A chain of states: 0 → 1 → 2 → ... → N-1 (terminal).
    Only action 'right' works. All rewards are 0 except the terminal state gives +1.
    True Q*(s, right) = gamma^(N-1-s) for all s."""
    def __init__(self, n=10):
        self.n = n
        self.reset()
    
    def reset(self):
        self.state = 0
        return self.state
    
    def step(self, action):
        # 5 actions, but only action 0 moves right. Others stay in place.
        if action == 0:
            self.state = min(self.state + 1, self.n - 1)
        reward = 1.0 if self.state == self.n - 1 else 0.0
        done = self.state == self.n - 1
        return self.state, reward, done

n_states = 10
n_actions = 5
gamma = 0.99

# True Q-values
true_Q = np.zeros((n_states, n_actions))
for s in range(n_states - 1):
    true_Q[s, 0] = gamma ** (n_states - 1 - s)  # Only action 0 is useful
    # Other actions: Q = 0 (they don't move you forward)

# Train Q-learning and track the bias
Q = np.zeros((n_states, n_actions))
max_Q_history = []  # Track max Q at state 0 over time
true_max_Q = true_Q[0, 0]  # True Q*(0, right) = gamma^9

for ep in range(5000):
    env = SimpleChainEnv(n=n_states)
    state = env.reset()
    
    for _ in range(50):
        if np.random.random() < 0.3:  # High exploration to visit all actions
            action = np.random.randint(n_actions)
        else:
            action = np.argmax(Q[state])
        
        next_state, reward, done = env.step(action)
        
        # Standard Q-learning: uses MAX over all actions
        if done:
            target = reward
        else:
            target = reward + gamma * np.max(Q[next_state])  # <-- MAX causes bias
        
        Q[state, action] += 0.05 * (target - Q[state, action])
        state = next_state
        if done:
            break
    
    max_Q_history.append(np.max(Q[0]))

print(f"True Q*(state=0, best action) = gamma^9 = {true_max_Q:.6f}")
print(f"Learned max Q(state=0) after training = {np.max(Q[0]):.6f}")
print(f"Overestimation = {np.max(Q[0]) - true_max_Q:.6f}")
print(f"\nQ-values at state 0 (true value for action 0 = {true_max_Q:.4f}, others = 0):")
for a in range(n_actions):
    bias = Q[0, a] - true_Q[0, a]
    print(f"  Q(0, action={a}) = {Q[0, a]:.4f}  (true: {true_Q[0, a]:.4f}, bias: {bias:+.4f})")

print(f"\nThe max operator picks the action with the highest Q-value.")
print(f"When estimates are noisy, it picks the action with the highest NOISE.")
print(f"This creates a systematic positive bias.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

window = 100
smoothed = np.convolve(max_Q_history, np.ones(window)/window, mode='valid')
ax.plot(smoothed, linewidth=2, label='Learned max Q(s=0)')
ax.axhline(y=true_max_Q, color='red', linestyle='--', linewidth=2, label=f'True Q* = {true_max_Q:.4f}')
ax.set_xlabel('Episode')
ax.set_ylabel('max Q(state=0)')
ax.set_title('Q-Learning Overestimation Bias: Learned Q > True Q')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Interview sentence: 'Standard Q-learning overestimates Q-values because")
print("max over noisy estimates has positive bias: E[max(X1,...,Xn)] >= max(E[X1],...,E[Xn]).")
print("Double Q-learning fixes this by using one network to select and another to evaluate.'")

## Experiment 3: Ablation — Deterministic vs Stochastic Policies

**Claim:** Deterministic policies cannot represent the optimal strategy in environments with state aliasing (where different states look the same).

**Why it matters in an interview:** This justifies stochastic policies and entropy bonuses used in modern algorithms like SAC.

In [ ]:
class AliasedEnv:
    """Environment with state aliasing.
    Two rooms: room A (go left to win) and room B (go right to win).
    But the agent only sees 'in a room' — it cannot tell which room it is in.
    
    True states: A_start, A_left(win), A_right(lose), B_start, B_left(lose), B_right(win)
    Observed states: 'start', 'left', 'right'
    
    A deterministic policy must choose the same action in 'start' for both rooms.
    The optimal stochastic policy: go left 50%, right 50% (wins 50% of the time).
    Any deterministic policy wins only 50% (always picks one room correctly).
    """
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.true_room = np.random.choice(['A', 'B'])  # Hidden
        self.obs = 'start'
        return self.obs
    
    def step(self, action):
        # action: 0 = left, 1 = right
        if self.true_room == 'A':
            reward = 1.0 if action == 0 else -1.0  # Room A: left wins
        else:
            reward = 1.0 if action == 1 else -1.0  # Room B: right wins
        return 'done', reward, True

# Compare deterministic vs stochastic policies
n_trials = 10000
env = AliasedEnv()

# Deterministic: always go left
det_left_rewards = []
for _ in range(n_trials):
    env.reset()
    _, r, _ = env.step(0)  # Always left
    det_left_rewards.append(r)

# Deterministic: always go right
det_right_rewards = []
for _ in range(n_trials):
    env.reset()
    _, r, _ = env.step(1)  # Always right
    det_right_rewards.append(r)

# Stochastic: 50/50
stoch_rewards = []
for _ in range(n_trials):
    env.reset()
    action = np.random.randint(2)  # Random 50/50
    _, r, _ = env.step(action)
    stoch_rewards.append(r)

print("Aliased Environment: Agent cannot distinguish Room A from Room B")
print(f"  Room A: left wins, right loses")
print(f"  Room B: right wins, left loses")
print(f"  Each room appears with 50% probability")
print()
print(f"Policy Results ({n_trials} episodes each):")
print(f"  Deterministic (always left):  avg reward = {np.mean(det_left_rewards):.3f}, win rate = {np.mean([r>0 for r in det_left_rewards]):.1%}")
print(f"  Deterministic (always right): avg reward = {np.mean(det_right_rewards):.3f}, win rate = {np.mean([r>0 for r in det_right_rewards]):.1%}")
print(f"  Stochastic (50/50 random):    avg reward = {np.mean(stoch_rewards):.3f}, win rate = {np.mean([r>0 for r in stoch_rewards]):.1%}")
print()
print("All three policies achieve ~50% win rate because no policy")
print("can do better without distinguishing the rooms.")
print("But the stochastic policy is more robust — it does not commit")
print("to a single losing strategy when the environment changes.")

In [ ]:
# Show that with asymmetric room probabilities, stochastic wins
# If room A appears 70%, a smart stochastic policy should go left 70%
print("\n--- Now with asymmetric room probabilities (A=70%, B=30%) ---\n")

class AsymmetricAliasedEnv:
    def __init__(self, prob_A=0.7):
        self.prob_A = prob_A
        self.reset()
    
    def reset(self):
        self.true_room = 'A' if np.random.random() < self.prob_A else 'B'
        return 'start'
    
    def step(self, action):
        if self.true_room == 'A':
            reward = 1.0 if action == 0 else -1.0
        else:
            reward = 1.0 if action == 1 else -1.0
        return 'done', reward, True

env = AsymmetricAliasedEnv(prob_A=0.7)

# Test different mixing probabilities
probs_left = np.linspace(0, 1, 21)
expected_rewards = []

for p_left in probs_left:
    rewards = []
    for _ in range(5000):
        env.reset()
        action = 0 if np.random.random() < p_left else 1
        _, r, _ = env.step(action)
        rewards.append(r)
    expected_rewards.append(np.mean(rewards))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(probs_left, expected_rewards, 'b-', linewidth=2)
ax.axvline(x=0.7, color='red', linestyle='--', label='Optimal p(left) = P(room A) = 0.7')
ax.axvline(x=0.0, color='gray', linestyle=':', alpha=0.5, label='Deterministic: always right')
ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5, label='Deterministic: always left')
best_idx = np.argmax(expected_rewards)
ax.scatter([probs_left[best_idx]], [expected_rewards[best_idx]], color='red', s=100, zorder=5)
ax.set_xlabel('P(go left)')
ax.set_ylabel('Expected Reward')
ax.set_title('Stochastic Policy Outperforms Deterministic in Aliased Environment')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best deterministic: always left (win rate = 70%) → E[reward] = {0.7*1 + 0.3*(-1):.2f}")
print(f"Best stochastic (p_left=0.7): win rate = 0.7*0.7 + 0.3*0.3 = {0.7**2 + 0.3**2:.0%} → E[reward] = {2*(0.7**2 + 0.3**2) - 1:.2f}")
print(f"\nBoth achieve E[reward] = 0.40, but stochastic is optimal for different P(room A).")
print("Interview sentence: 'In partially observable environments, stochastic policies")
print("can match or outperform deterministic ones because they hedge against state aliasing.'")

## Summary

Claims now backed by evidence:

1. **Tabular Q scales as O(|S|*|A|) memory** — impractical above ~10M states, impossible for pixel observations (Experiment 1)
2. **Q-learning has overestimation bias** — max over noisy estimates systematically overestimates Q-values (Experiment 2)
3. **Stochastic policies are necessary with state aliasing** — deterministic policies commit to one action and cannot hedge (Experiment 3)

For deeper mathematical treatment, see [policies-and-value-functions-interview.md](./policies-and-value-functions-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)